# NLP Assignment 3

# Part A

In [ ]:
import math
import re
from collections import Counter
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
torch.manual_seed(42)

DATA_FILES = {
    "train": "train.csv",
    "val": "val.csv",
    "test": "test.csv",
}

MAX_LEN = 128
MIN_FREQ = 2
BATCH_SIZE = 64
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

Using device: cpu


##  create labels

In [26]:
def rating_to_sentiment(rating: int) -> int:
    #    maps ratings into the required 3 sentiment classes.
    if rating <= 2:
        return 0  # Negative
    if rating == 3:
        return 1  # Neutral
    return 2      # Positive


def load_split(path: str) -> pd.DataFrame:
    #    loads one CSV split and keeps only needed columns.
    df = pd.read_csv(path)
    df = df[["review_text", "rating", "category"]].copy()
    df["rating"] = df["rating"].astype(int)
    df["sentiment_label"] = df["rating"].apply(rating_to_sentiment)
    return df


splits = {name: load_split(path) for name, path in DATA_FILES.items()}

for name, df in splits.items():
    print(f"{name}: {len(df)} samples")

splits["train"].head(2)

train: 25200 samples
val: 5400 samples
test: 5400 samples


,review_text,rating,category,sentiment_label
0,Love this spray!! Hold and great shine all in ...,5,Beauty,2
1,I got the 16-ga 100' spool. It arrived on tim...,5,Electronics,2


## Tokenize text and build vocabulary from training data only

In [27]:
def clean_text(text: str) -> str:
    #    lightly normalizes text to make tokenization consistent.
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize(text: str) -> list[str]:
    #    splits text into basic word-like tokens.
    text = clean_text(text)
    return re.findall(r"[a-z0-9']+", text)


def build_vocab(train_texts: pd.Series, min_freq: int = 2) -> tuple[dict[str, int], dict[int, str]]:
    #    builds a vocabulary only from training split tokens.
    counter = Counter()
    for text in train_texts:
        counter.update(tokenize(text))

    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    itos = {idx: token for token, idx in vocab.items()}
    return vocab, itos


def review_length(text: str) -> int:
    #    counts tokens in a review after cleaning.
    return len(tokenize(text))


def add_length_label(splits: dict[str, pd.DataFrame]) -> int:
    #    adds a short/long label using the training median length threshold.
    train_lengths = splits["train"]["review_text"].apply(review_length)
    threshold = int(train_lengths.median())

    for split_name, df in splits.items():
        lengths = df["review_text"].apply(review_length)
        df["length_label"] = (lengths >= threshold).astype(int)
        print(f"{split_name} length label counts: {df['length_label'].value_counts().to_dict()}")

    return threshold


length_threshold = add_length_label(splits)
vocab, itos = build_vocab(splits["train"]["review_text"], min_freq=MIN_FREQ)
print(f"Length threshold (train median tokens): {length_threshold}")
print(f"Vocab size (train only): {len(vocab)}")
print("Sample tokens:", list(vocab.keys())[2:12])

train length label counts: {1: 12769, 0: 12431}
val length label counts: {1: 2746, 0: 2654}
test length label counts: {1: 2781, 0: 2619}
Length threshold (train median tokens): 58
Vocab size (train only): 22827
Sample tokens: ['love', 'this', 'spray', 'hold', 'and', 'great', 'shine', 'all', 'in', 'one']


## Convert tokens to indices and apply fixed padding 

In [28]:
def tokens_to_indices(tokens: list[str], vocab: dict[str, int]) -> list[int]:
    #    changes tokens into integer ids using the vocabulary.
    return [vocab.get(tok, UNK_IDX) for tok in tokens]


def pad_or_truncate(ids: list[int], max_len: int, pad_idx: int = PAD_IDX) -> tuple[list[int], list[int]]:
    #    makes every sequence the same fixed length.
    if len(ids) >= max_len:
        trimmed = ids[:max_len]
        mask = [1] * max_len
        return trimmed, mask

    pad_count = max_len - len(ids)
    padded = ids + [pad_idx] * pad_count
    mask = [1] * len(ids) + [0] * pad_count
    return padded, mask


def preprocess_split(df: pd.DataFrame, vocab: dict[str, int], max_len: int) -> dict[str, torch.Tensor]:
    #    preprocesses one split into tensors for model input.
    input_ids, attention_masks = [], []

    for text in df["review_text"]:
        tokens = tokenize(text)
        token_ids = tokens_to_indices(tokens, vocab)
        ids_fixed, mask_fixed = pad_or_truncate(token_ids, max_len=max_len)
        input_ids.append(ids_fixed)
        attention_masks.append(mask_fixed)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "sentiment_label": torch.tensor(df["sentiment_label"].to_numpy(), dtype=torch.long),
        "length_label": torch.tensor(df["length_label"].to_numpy(), dtype=torch.long),
    }


preprocessed = {
    split_name: preprocess_split(df, vocab=vocab, max_len=MAX_LEN)
    for split_name, df in splits.items()
}

for split_name, data_dict in preprocessed.items():
    print(
        split_name,
        data_dict["input_ids"].shape,
        data_dict["attention_mask"].shape,
        data_dict["sentiment_label"].shape,
        data_dict["length_label"].shape,
    )

train torch.Size([25200, 128]) torch.Size([25200, 128]) torch.Size([25200]) torch.Size([25200])
val torch.Size([5400, 128]) torch.Size([5400, 128]) torch.Size([5400]) torch.Size([5400])
test torch.Size([5400, 128]) torch.Size([5400, 128]) torch.Size([5400]) torch.Size([5400])


## Build dataloaders from the preprocessed tensors

In [29]:
def build_dataloader(split_dict: dict[str, torch.Tensor], batch_size: int, shuffle: bool) -> DataLoader:
    #    wraps processed tensors into a PyTorch dataloader.
    ds = TensorDataset(
        split_dict["input_ids"],
        split_dict["attention_mask"],
        split_dict["sentiment_label"],
        split_dict["length_label"],
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


loaders = {
    "train": build_dataloader(preprocessed["train"], BATCH_SIZE, shuffle=True),
    "val": build_dataloader(preprocessed["val"], BATCH_SIZE, shuffle=False),
    "test": build_dataloader(preprocessed["test"], BATCH_SIZE, shuffle=False),
}

batch = next(iter(loaders["train"]))
print("Batch tensor shapes:", [x.shape for x in batch])

Batch tensor shapes: [torch.Size([64, 128]), torch.Size([64, 128]), torch.Size([64]), torch.Size([64])]


## Implement Transformer Encoder 

In [30]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        #    precomputes sinusoidal positions once for fast reuse.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #    adds position information to token embeddings.
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        #    checks dimensions and prepares projection layers.
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        #    reshapes vectors into independent attention heads.
        batch_size, seq_len, _ = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def _combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        #    merges all heads back into one vector per token.
        batch_size, _, seq_len, _ = x.size()
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        #    computes self-attention using Q, K, and V projections.
        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = torch.matmul(attn_weights, v)
        context = self._combine_heads(context)
        return self.out_proj(context)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        #    builds a small two-layer network for each token.
        self.net = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        #    applies the feed-forward network token by token.
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        #    creates one encoder block with attention and feed-forward parts.
        self.self_attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, ff_dim, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor | None = None) -> torch.Tensor:
        #    applies residual connections and layer normalization in order.
        attn_out = self.self_attn(x, attention_mask)
        x = self.norm1(x + self.dropout1(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        return x


class EncoderTransformer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        num_heads: int = 4,
        ff_dim: int = 256,
        num_layers: int = 2,
        max_len: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()
        #    assembles token embedding, positional encoding, and encoder stack.
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.position_encoding = PositionalEncoding(d_model=d_model, max_len=max_len)
        self.dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList(
            [EncoderBlock(d_model, num_heads, ff_dim, dropout) for _ in range(num_layers)]
        )

    def masked_mean_pool(self, x: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        #    computes one fixed review vector by averaging valid tokens only.
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(x * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / counts

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        #    returns token-level outputs and one pooled review embedding.
        x = self.token_embedding(input_ids)
        x = self.position_encoding(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, attention_mask)

        pooled = self.masked_mean_pool(x, attention_mask)
        return x, pooled

##  encoder model 

In [ ]:
encoder_model = EncoderTransformer(
    vocab_size=len(vocab),
    d_model=128,
    num_heads=4,
    ff_dim=256,
    num_layers=2,
    max_len=MAX_LEN,
    dropout=0.1,
)
sample_input_ids, sample_mask, _, _ = next(iter(loaders["train"]))

with torch.no_grad():
    token_outputs, review_embeddings = encoder_model(sample_input_ids, sample_mask)

print("Token outputs shape:", token_outputs.shape)    
print("Review embeddings shape:", review_embeddings.shape) 

Token outputs shape: torch.Size([64, 128, 128])
Review embeddings shape: torch.Size([64, 128])
